In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import mean_squared_error, mean_absolute_error
import json
import os

# Load processed data
train_df = pd.read_csv('../data/processed/train_processed.csv')
test_df  = pd.read_csv('../data/processed/test_processed.csv')
rul_df   = pd.read_csv('../data/processed/rul_fd001.csv')

with open('../data/processed/useful_sensors.json') as f:
    useful_sensors = json.load(f)

print("✅ Data loaded!")
print("Train shape:", train_df.shape)
print("TensorFlow version:", tf.__version__)

✅ Data loaded!
Train shape: (20631, 27)
TensorFlow version: 2.21.0


In [2]:
# Create sliding window sequences for LSTM
# LSTM needs sequences of past cycles to predict RUL

SEQUENCE_LENGTH = 30  # look back 30 cycles
feature_cols = useful_sensors + ['cycle']

def create_sequences(df, seq_len, feature_cols):
    X, y = [], []
    for engine_id in df['engine_id'].unique():
        engine_data = df[df['engine_id'] == engine_id].reset_index(drop=True)
        features = engine_data[feature_cols].values
        rul      = engine_data['RUL'].values
        for i in range(seq_len, len(engine_data)):
            X.append(features[i-seq_len:i])
            y.append(rul[i])
    return np.array(X), np.array(y)

print("⏳ Creating sequences... please wait...")
X_train, y_train = create_sequences(train_df, SEQUENCE_LENGTH, feature_cols)

# Prepare test sequences (last 30 cycles of each engine)
def create_test_sequences(df, seq_len, feature_cols):
    X = []
    for engine_id in sorted(df['engine_id'].unique()):
        engine_data = df[df['engine_id'] == engine_id]
        features = engine_data[feature_cols].values
        if len(features) >= seq_len:
            X.append(features[-seq_len:])
        else:
            # Pad if engine has fewer cycles than sequence length
            pad = np.zeros((seq_len - len(features), len(feature_cols)))
            X.append(np.vstack([pad, features]))
    return np.array(X)

X_test = create_test_sequences(test_df, SEQUENCE_LENGTH, feature_cols)
y_test = rul_df['RUL'].values

print(f"✅ Sequences created!")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")

⏳ Creating sequences... please wait...
✅ Sequences created!
X_train shape: (17631, 30, 15)
y_train shape: (17631,)
X_test shape:  (100, 30, 15)


In [3]:
# Build LSTM Model
model = Sequential([
    LSTM(128, return_sequences=True, 
         input_shape=(SEQUENCE_LENGTH, len(feature_cols))),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, 
                            restore_best_weights=True)
checkpoint = ModelCheckpoint('../models/lstm_best.keras', 
                              save_best_only=True, monitor='val_loss')

print("\n⏳ Training LSTM... this will take 3-5 minutes...")
print("You'll see loss decreasing each epoch — that means it's learning!\n")

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print("\n✅ LSTM Training Complete!")

c:\Users\Diya patel\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 128)        │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,249 (489.25 KB)

 Trainable params: 125,249 (489.25 KB)

 Non-trainable params: 0 (0.00 B)


⏳ Training LSTM... this will take 3-5 minutes...
You'll see loss decreasing each epoch — that means it's learning!

Epoch 1/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 11s 133ms/step - loss: 6540.6768 - mae: 69.9902 - val_loss: 5264.0459 - val_mae: 62.7567
Epoch 2/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 7s 114ms/step - loss: 3289.4351 - mae: 48.8257 - val_loss: 2454.0972 - val_mae: 44.1854
Epoch 3/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 8s 124ms/step - loss: 1904.0258 - mae: 38.8548 - val_loss: 1811.2290 - val_mae: 38.5108
Epoch 4/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 8s 121ms/step - loss: 1550.7289 - mae: 34.6875 - val_loss: 1692.9196 - val_mae: 35.6204
Epoch 5/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 7s 114ms/step - loss: 906.0568 - mae: 25.2721 - val_loss: 1580.5054 - val_mae: 31.7484
Epoch 6/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 10s 153ms/step - loss: 716.9495 - mae: 21.3074 - val_loss: 1354.2234 - val_mae: 28.8277
Epoch 7/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 12s 175ms/step - loss: 510.2820 - mae: 17.6608 - val_loss: 597.9119 - val_mae: 20.0069
Ep

In [4]:
# Evaluate LSTM on test data
lstm_preds = model.predict(X_test).flatten().clip(0, 125)
lstm_rmse = np.sqrt(mean_squared_error(y_test, lstm_preds))
lstm_mae = mean_absolute_error(y_test, lstm_preds)

print("🏆 FINAL MODEL COMPARISON:")
print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8}")
print("-" * 40)
print(f"{'Linear Regression':<20} {'22.43':>8} {'17.64':>8}")
print(f"{'Random Forest':<20} {'17.93':>8} {'13.22':>8}")
print(f"{'XGBoost':<20} {'18.72':>8} {'14.06':>8}")
print(f"{'LSTM':<20} {lstm_rmse:>8.2f} {lstm_mae:>8.2f}")

# Save LSTM predictions
import json
with open('../data/processed/predictions.json') as f:
    preds_data = json.load(f)

preds_data['lstm_preds'] = lstm_preds.tolist()
preds_data['lstm_rmse'] = lstm_rmse
preds_data['lstm_mae'] = lstm_mae

with open('../data/processed/predictions.json', 'w') as f:
    json.dump(preds_data, f)

# Save model
model.save('../models/lstm_model.keras')
print("\n✅ LSTM predictions saved!")
print("✅ LSTM model saved!")

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step
🏆 FINAL MODEL COMPARISON:
Model                    RMSE      MAE
----------------------------------------
Linear Regression       22.43    17.64
Random Forest           17.93    13.22
XGBoost                 18.72    14.06
LSTM                    14.33    10.43

✅ LSTM predictions saved!
✅ LSTM model saved!


In [5]:
# Calculate accuracy percentage for all models
# We define accuracy as: predictions within ±13 cycles of actual RUL

def calculate_accuracy(actual, predicted, tolerance=13):
    actual = np.array(actual)
    predicted = np.array(predicted)
    within_tolerance = np.abs(actual - predicted) <= tolerance
    accuracy = (np.sum(within_tolerance) / len(actual)) * 100
    return accuracy

lr_acc   = calculate_accuracy(y_test, lr_preds)
rf_acc   = calculate_accuracy(y_test, rf_preds)
xgb_acc  = calculate_accuracy(y_test, xgb_preds)
lstm_acc = calculate_accuracy(y_test, lstm_preds)

print("🏆 FINAL MODEL COMPARISON WITH ACCURACY:")
print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8} {'Accuracy':>10}")
print("-" * 52)
print(f"{'Linear Regression':<20} {'22.43':>8} {'17.64':>8} {lr_acc:>9.1f}%")
print(f"{'Random Forest':<20} {'17.93':>8} {'13.22':>8} {rf_acc:>9.1f}%")
print(f"{'XGBoost':<20} {'18.72':>8} {'14.06':>8} {xgb_acc:>9.1f}%")
print(f"{'LSTM':<20} {14.33:>8} {10.43:>8} {lstm_acc:>9.1f}%")

# Save accuracy to predictions.json
with open('../data/processed/predictions.json') as f:
    preds_data = json.load(f)

preds_data['accuracies'] = {
    'Linear Regression': round(lr_acc, 1),
    'Random Forest':     round(rf_acc, 1),
    'XGBoost':           round(xgb_acc, 1),
    'LSTM':              round(lstm_acc, 1)
}

with open('../data/processed/predictions.json', 'w') as f:
    json.dump(preds_data, f)

print("\n✅ Accuracies saved!")

NameError: name 'lr_preds' is not defined

In [6]:
import numpy as np
import json

# Load all predictions from saved file
with open('../data/processed/predictions.json') as f:
    preds_data = json.load(f)

y_test    = np.array(preds_data['actual_rul'])
lr_preds  = np.array(preds_data['lr_preds'])
rf_preds  = np.array(preds_data['rf_preds'])
xgb_preds = np.array(preds_data['xgb_preds'])
lstm_preds = np.array(preds_data['lstm_preds'])

# Calculate accuracy percentage for all models
def calculate_accuracy(actual, predicted, tolerance=13):
    actual = np.array(actual)
    predicted = np.array(predicted)
    within_tolerance = np.abs(actual - predicted) <= tolerance
    accuracy = (np.sum(within_tolerance) / len(actual)) * 100
    return accuracy

lr_acc   = calculate_accuracy(y_test, lr_preds)
rf_acc   = calculate_accuracy(y_test, rf_preds)
xgb_acc  = calculate_accuracy(y_test, xgb_preds)
lstm_acc = calculate_accuracy(y_test, lstm_preds)

print("🏆 FINAL MODEL COMPARISON WITH ACCURACY:")
print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8} {'Accuracy':>10}")
print("-" * 52)
print(f"{'Linear Regression':<20} {'22.43':>8} {'17.64':>8} {lr_acc:>9.1f}%")
print(f"{'Random Forest':<20} {'17.93':>8} {'13.22':>8} {rf_acc:>9.1f}%")
print(f"{'XGBoost':<20} {'18.72':>8} {'14.06':>8} {xgb_acc:>9.1f}%")
print(f"{'LSTM':<20} {'14.33':>8} {'10.43':>8} {lstm_acc:>9.1f}%")

# Save accuracies
preds_data['accuracies'] = {
    'Linear Regression': round(lr_acc, 1),
    'Random Forest':     round(rf_acc, 1),
    'XGBoost':           round(xgb_acc, 1),
    'LSTM':              round(lstm_acc, 1)
}

with open('../data/processed/predictions.json', 'w') as f:
    json.dump(preds_data, f)

print("\n✅ Accuracies saved!")

🏆 FINAL MODEL COMPARISON WITH ACCURACY:
Model                    RMSE      MAE   Accuracy
----------------------------------------------------
Linear Regression       22.43    17.64      45.0%
Random Forest           17.93    13.22      60.0%
XGBoost                 18.72    14.06      59.0%
LSTM                    14.33    10.43      71.0%

✅ Accuracies saved!


In [7]:
def calculate_accuracy(actual, predicted, tolerance=20):
    actual = np.array(actual)
    predicted = np.array(predicted)
    within_tolerance = np.abs(actual - predicted) <= tolerance
    accuracy = (np.sum(within_tolerance) / len(actual)) * 100
    return accuracy

print("With ±20 cycle tolerance:")
print(f"Linear Regression: {calculate_accuracy(y_test, lr_preds):.1f}%")
print(f"Random Forest:     {calculate_accuracy(y_test, rf_preds):.1f}%")
print(f"XGBoost:           {calculate_accuracy(y_test, xgb_preds):.1f}%")
print(f"LSTM:              {calculate_accuracy(y_test, lstm_preds):.1f}%")

With ±20 cycle tolerance:
Linear Regression: 67.0%
Random Forest:     77.0%
XGBoost:           72.0%
LSTM:              79.0%


In [8]:
with open('../data/processed/predictions.json') as f:
    preds_data = json.load(f)

preds_data['accuracies_strict'] = {
    'Linear Regression': 45.0,
    'Random Forest': 60.0,
    'XGBoost': 59.0,
    'LSTM': 71.0
}

preds_data['accuracies_industry'] = {
    'Linear Regression': 67.0,
    'Random Forest': 77.0,
    'XGBoost': 72.0,
    'LSTM': 79.0
}

with open('../data/processed/predictions.json', 'w') as f:
    json.dump(preds_data, f)

print("✅ Both accuracy metrics saved!")

✅ Both accuracy metrics saved!
